# Week 7: Getting Data with Web APIs
### UN5550 — Introduction to Data Science

**Lecture objective:** Request real data from an API, understand its JSON response, and turn it into a useful pandas table.

### Lecture Outline
- API fundamentals and request parameters.
- JSON parsing and nested data.
- Pokémon data and GitHub pagination.
- Houghton weather and World Bank indicators.
- Error handling, authentication, and reproducibility.

All examples use **real HTTP requests** and require internet access, but no API key. Results may change.

In [ ]:
import requests
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl 
%matplotlib inline  

import ssl
ssl._create_default_https_context = ssl._create_unverified_context

---
## JSON
JSON is a text format used to exchange structured data. After Python parses it, JSON objects become dictionaries and JSON arrays become lists.

### JSON data types

| Type | Description |
| --- | --- |
| String | Anything inside double quotes. |
| Number | Any number (no difference between ints and floats). |
| Boolean | `true` and `false`. |
| Null | JSON's empty value, denoted by `null`. |
| Array | Like Python lists. |
| Object | A collection of key-value pairs, like dictionaries. Keys must be strings, values can be anything (even other objects). |

See [json-schema.org](https://json-schema.org/understanding-json-schema/reference/type.html) for more details.

### Example JSON object

See `data/family.json`.

<center><img src='imgs/hierarchy.png' width=50%></center>

In [ ]:
import json
from pathlib import Path

f = Path('data') / 'family.json'
family_tree = json.loads(f.read_text())
family_tree

In [ ]:
# We can access the nested information 
family_tree['children']

In [ ]:
family_tree['children'][1]['children'][0]['age']

In [ ]:
family_tree['children'][0]['children']

### Reading and writing JSON
- `json.loads(text)` converts JSON text into Python objects.
- `json.dumps(object)` converts Python objects into JSON text.
- `response.json()` parses JSON returned by an HTTP request.

`None` represents a missing value here; it is not zero. Never use `eval()` to parse JSON.

**Predict:** What type is `data['observations']`? How would you access Tuesday's day name?

### Example: loads( )
We can parse JSON using Python’s `json` module. In particular, we will use its `loads` function, which deserializes a string representing a JSON object into a Python object.

In [ ]:
serialized = """{ "title" : "Data Science Book",
                  "author" : "Joel Grus",
                  "publicationYear" : 2014,
                  "topics" : [ "data", "science", "data science"] }"""

# parse the JSON to create a Python dict
deserialized = json.loads(serialized)
deserialized

We can then use the deserialized object like a Python dict object to find information.  For example, does the book cover the topic of "data science":

In [ ]:
type(serialized)

In [ ]:
type(deserialized)

In [ ]:
if "data science" in deserialized["topics"]:
    print("Yes")
else: 
    print("No")

In [ ]:
deserialized.keys()

### Example: dumps( )

The `dumps` function will take a Python object (e.g., dict) and serializes it into a JSON formatted string. 

In [ ]:
data = {
   'name' : 'ACME',
   'shares' : 100,
   'price' : 542.23
}
json_obj = json.dumps(data)
json_obj

In [ ]:
type(data)

In [ ]:
type(json_obj)

In [ ]:
data2 = json.loads(json_obj)
data2

In [ ]:
type(data2)

In [ ]:
help(json.dumps)

### Example  
*Ref: https://pythonspot.com/en/json-encoding-and-decoding-with-python/*

In [ ]:
# Convert JSON to Python Object, then iterate
array = '{"drinks": ["coffee", "tea", "water"]}'
data = json.loads(array)
 
for element in data['drinks']:
    print(element)

In [ ]:
json_data = '{"name": "Brian", "city": "Seattle"}'
python_obj = json.loads(json_data)
print(python_obj["name"])
print(python_obj["city"])

In [ ]:
obj = {
    "persons": [
        {
            "city": "Seattle", 
            "name": "Brian"
        }, 
        {
            "city": "Amsterdam", 
            "name": "David"
        }
    ]
}
obj

In [ ]:
obj['persons']

In [ ]:
obj['persons'][0]

In [ ]:
obj['persons'][0]['city']

In [ ]:
obj.keys()

In [ ]:
obj['persons'][0].keys()

---
# Aside:  lambda and map functions

A lambda function is a small anonymous function that can take any number of arguments, but can only have one expression. 

It has the following syntax: 

`lambda` *arguments* : *expression*

The expression is executed and the result returned.

#### Example   
A lambda function that adds 10 to the number passed in as an argument.

In [ ]:
# Example 
x = lambda a: a+10
print(x(5))

#### Example   
A lambda function that takes two inputs and multiplies them together.

In [ ]:
# Example 
x = lambda a, b : a*b
print(x(5,6))

#### Example  
Apply the lambda function to an argument by surrounding the function and argument in parentheses:

In [ ]:
(lambda x: x + 1)(2)

#### Example  
lambda function are often used with other methods in Python such as `apply`, `filter`, `map`, `sorted`, etc.

In [ ]:
ids = ['id1', 'id2', 'id30', 'id3', 'id22', 'id100']
print(sorted(ids)) # Lexicographic sort

In [ ]:
sorted_ids = sorted(ids, key=lambda x: int(x[2:])) # Integer sort
print(sorted_ids)

#### Example   
Here is an example using the `map` function, which expects a function object and any number of iterables, such as a list, dictionary, etc.   `map` executes the function_object for each element in the sequence and returns a list of the elements modified by the function object. 

In [ ]:
def multiply2(x): 
    return x * 2 

x = map(multiply2, [1, 2, 3, 4])
print(x)

In [ ]:
def print_iterator(it):
    for x in it:
        print(x, end=' ')
    print('')

print_iterator(x)

In [ ]:
mp_it = map(lambda x : x * 2, [1, 2, 3, 4])
print_iterator(mp_it)

In [ ]:
list_numbers = [1, 2, 3, 4]
tuple_numbers = (5, 6, 7, 8)
map_iterator = map(lambda x, y: x * y, list_numbers, tuple_numbers)
print_iterator(map_iterator)

---
# API

## 1. What is an API?
A web API is an interface through which your Python program requests data from a service.

For example, a weather website displays a forecast to people. Its API can return temperature values directly to your program. We can then analyze those values without extracting them from the webpage's HTML.

**Web scraping:** Download a webpage and locate data in HTML.

**API access:** Request a documented resource and read structured fields.

An API is useful when you need specific variables, repeated collection, filtering, or many records. Its data still need validation: an API response is not automatically complete or correct.

### How a request works
1. Python sends a request to an **endpoint**, a URL for a particular resource.
2. The service processes the request.
3. It returns a **response** containing a status code and data.
4. Python extracts the data, often from JSON.

We will use `GET` to retrieve data. `POST` sends data for an operation or creates a resource; its meaning depends on the endpoint.

**Discuss:** Why might an API be easier to maintain than a scraper when a website changes its layout?

### API requests

- API requests are just `GET`/`POST` requests to a specially maintained URL.
- Let's test out the [Pokémon API](https://pokeapi.co).

First, let's make a `GET` request for `'squirtle'`. To do this, we need to make a request to the correct URL.

In [ ]:
def create_url(pokemon):
    return f'https://pokeapi.co/api/v2/pokemon/{pokemon}'

create_url('squirtle')

In [ ]:
r = requests.get(create_url('squirtle'))
r

Remember, the 200 status code is good! Let's take a look at the **content**:

In [ ]:
r.content[:1000]

Looks like JSON. We can extract the JSON from this request with the `json` method (or by passing `r.text` to `json.loads`).

In [ ]:
rr = r.json()
rr

In [ ]:
rr.keys()

In [ ]:
rr['weight']

In [ ]:
rr['abilities'][1]['ability']['name']

Let's try a `GET` request for `billy`

In [ ]:
r = requests.get(create_url('billy'))
r

We receive a 404 error, since there is no Pokemon named `'billy'`!

---
### API Example 1: NASA 

Let's try using the [OpenNotify API](http://open-notify.org/) that serves NASA data. 

Let's use the GET request to see what data we can get in response. Get request takes the URL, in our case the url to Open Notify. Lets make a request and print what is returned. When we just make a request to a url without the right endpoint, we get the html content as response. **End points** are locations of the resources.

In [ ]:
request = requests.get('http://api.open-notify.org')
print(request.text)

In [ ]:
print(request.status_code)

Let's try to request something from the API for an end point that doesn't exist. 

In [ ]:
request2 = requests.get('http://api.open-notify.org/fake-endpoint')
print(request2.status_code)

We get a 404 Error.  

Now let's try for a real end point.  For example, we can get the current location to the International Space Station, with the endpoint `/iss-now.json`.  Alternatively, `/iss-pass.json` returns the time at which the space station passes overhead. 

In [ ]:
issLoc = requests.get('http://api.open-notify.org/iss-now.json')
print(issLoc.text)

obj = json.loads(issLoc.text)

print(obj['timestamp'])
print(obj['iss_position']['latitude'], obj['iss_position']['latitude'])

The API description tells use how to request and what the expected output will be. 

In this case the data returned has the following format: 

```
{
  "message": "success", 
  "timestamp": UNIX_TIME_STAMP, 
  "iss_position": {
    "latitude": CURRENT_LATITUDE, 
    "longitude": CURRENT_LONGITUDE
  }
}
```

Let's now convert the `UNIX_TIME_STAMP` to a time that is readable using the `datetime` module. 


In [ ]:
from datetime import datetime, timezone
print(datetime.utcfromtimestamp(obj['timestamp']).strftime('%Y-%m-%d %H:%M:%S'))

Another endpoint available gets information on the people in space. 

In [ ]:
people = requests.get('http://api.open-notify.org/astros.json')
print(people.text)

In [ ]:
people_json  = people.json()
print(people_json)

We can make use of the json `dumps()` function optional arguments to *pretty print* the JSON array elements and object members.

In [ ]:
people_json_obj = json.dumps(people.json(), indent=2)
print(people_json_obj)

In [ ]:
#To print the number of people in space
print("Number of people in space:",people_json['number'])
#To print the names of people in space using a for loop
for p in people_json['people']:
    print(p['name'])

---
## API Example2: GitHub parameters and pagination
[Repository endpoint](https://docs.github.com/en/rest/repos/repos) · [Pagination](https://docs.github.com/en/rest/using-the-rest-api/using-pagination-in-the-rest-api)

**Question:** Which languages appear in Joel Grus's recently created public repositories?

The endpoint `/users/joelgrus/repos` returns a list of repository records. Parameters control sorting and page size. `params=` encodes the query for us.

### Step 1 — Request five recently created repositories

In [ ]:
github_url = "https://api.github.com/users/joelgrus/repos"
params = {"sort": "created", "direction": "desc", "per_page": 5, "page": 1}
headers = {"Accept": "application/vnd.github+json"}
r = requests.get(github_url, params=params, headers=headers, timeout=20)
print("Requested URL:", r.url)
print("Status:", r.status_code)
print("Requests remaining:", r.headers.get("X-RateLimit-Remaining"))
r.raise_for_status()
repos = r.json()
print("Records on this page:", len(repos))

### Step 2 — Select useful fields
The response is a list of dictionaries, so access one repository using `repos[0]`, rather than calling `repos.keys()`.

A `null` language means no primary language is reported. It does not mean the repository contains no useful content.

In [ ]:
rows = [{
    "id": repo["id"], "name": repo["name"], "language": repo["language"],
    "stars": repo["stargazers_count"], "created_at": repo["created_at"],
    "fork": repo["fork"]
} for repo in repos]
repo_df = pd.DataFrame(rows)
if not repo_df.empty:
    repo_df["created_at"] = pd.to_datetime(repo_df["created_at"], utc=True)
display(repo_df)

### Step 3 — Collect more than one page
**Pagination** divides a collection across responses. A page of five records does not mean the user owns only five repositories.

GitHub can return a `Link` header with a next-page URL. Requests exposes that as `r.links`. In the following demonstration, collect at most three pages. This deliberate cap creates a partial dataset if more pages remain.

In [ ]:
all_repos = []
next_url = github_url
for page_number in range(1, 4):
    r = requests.get(
        next_url, params=params if page_number == 1 else None,
        headers=headers, timeout=20
    )
    r.raise_for_status()
    page = r.json()
    all_repos.extend(page)
    print("Page", page_number, "records:", len(page))
    next_url = r.links.get("next", {}).get("url")
    if next_url is None:
        break
print("Total retrieved:", len(all_repos))
print("More pages remain:", next_url is not None)

In [ ]:
repo_sample = pd.DataFrame([
    {"id": x["id"], "name": x["name"], "language": x["language"]} for x in all_repos
]).drop_duplicates("id")
languages = repo_sample["language"].fillna("Unknown / not reported").value_counts()
display(languages)
if not languages.empty:
    languages.plot.barh()
    plt.xlabel("Repository count")
    plt.title("Languages in the retrieved repository sample")
    plt.tight_layout()
    plt.show()

### Practice and interpretation
Change `sort` from `created` to `updated`. Explain what changes in the question.

**Discussion:** Recently created repositories differ from recently updated repositories. The first concerns new projects; the second concerns recent activity. A developer's public repositories do not capture private projects, and the reported primary language does not capture all languages used.

**Pagination check:** Before calling a dataset complete, confirm that no next page remains. Across changing data, duplicates or omissions can occur between page requests; stable IDs help detect duplicates.



---
## API Example3: Weather (real time-series data)
[Open-Meteo historical weather documentation](https://open-meteo.com/en/docs/historical-weather-api)

**Question:** How did daily maximum temperature and precipitation vary near Houghton during October 2025?

This example requests a fixed historical month. The archive provides reanalysis/model weather data; it should not be described as measurements from a particular local weather station.

### Step 1 — Specify location, date range, and variables
Latitude and longitude identify a location. `daily` selects variables. `timezone` defines the daily calendar periods.

In [ ]:
weather_url = "https://archive-api.open-meteo.com/v1/archive"
weather_params = {
    "latitude": 47.12, "longitude": -88.57,
    "start_date": "2025-10-01", "end_date": "2025-10-31",
    "daily": "temperature_2m_max,precipitation_sum",
    "timezone": "America/Detroit"
}
r = requests.get(weather_url, params=weather_params, timeout=30)
r.raise_for_status()
weather = r.json()
print("Response keys:", list(weather.keys()))
print("Units:", weather["daily_units"])
print("Timezone:", weather["timezone"])

### Step 2 — Understand the response structure
The `daily` object contains aligned arrays. The first date, first temperature, and first precipitation value describe the same day. This differs from GitHub's list of record dictionaries.

Inspect the first three entries before creating the table.

In [ ]:
daily = weather["daily"]
for key in ["time", "temperature_2m_max", "precipitation_sum"]:
    print(key, ":", daily[key][:3])
lengths = [len(daily[key]) for key in ["time", "temperature_2m_max", "precipitation_sum"]]
if len(set(lengths)) != 1:
    raise ValueError("Daily arrays have different lengths")

### Step 3 — Build a DataFrame
Each row represents one local calendar day at the selected location. Convert the date column and retain missing values.

In [ ]:
weather_df = pd.DataFrame(daily).rename(columns={
    "time": "date", "temperature_2m_max": "max_temperature",
    "precipitation_sum": "precipitation"
})
weather_df["date"] = pd.to_datetime(weather_df["date"])
display(weather_df.head())
print("Number of days:", len(weather_df))
print("Missing values:")
print(weather_df.isna().sum())

### Step 4 — Visualize and summarize
Use separate panels because temperature and precipitation have different units. Always read the units from the response rather than assume them.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(weather_df["date"], weather_df["max_temperature"], marker="o")
axes[0].set_ylabel("Max temperature (" + weather["daily_units"]["temperature_2m_max"] + ")")
axes[1].bar(weather_df["date"], weather_df["precipitation"])
axes[1].set_ylabel("Precipitation (" + weather["daily_units"]["precipitation_sum"] + ")")
axes[1].set_xlabel("Local date")
fig.suptitle("Houghton-area weather — October 2025")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
print("Average daily maximum:", weather_df["max_temperature"].mean())
print("Missing precipitation days:", weather_df["precipitation"].isna().sum())
print("Sum over available days:", weather_df["precipitation"].sum(min_count=1))

---
# Homework: Practice
## API Example4: World Bank country–year observations
[Documentation](https://datahelpdesk.worldbank.org/knowledgebase/articles/898581-api-basic-call-structures)

**Question:** What total population values are reported for Bangladesh and the United States from 2020 to 2023?

The indicator `SP.POP.TOTL` means total population. The country codes are `BGD` and `USA`. The semicolon requests both countries. `format=json` selects JSON.

### Step 1 — Request the indicator

In [ ]:
wb_url = "https://api.worldbank.org/v2/country/BGD;USA/indicator/SP.POP.TOTL"
wb_params = {"format": "json", "date": "2020:2023", "per_page": 100}
r = requests.get(wb_url, params=wb_params, timeout=30)
r.raise_for_status()
wb_data = r.json()
print("Top-level type:", type(wb_data).__name__)
print("Metadata:", wb_data[0])

### Step 2 — Separate metadata from observations
This API returns a two-element list: `wb_data[0]` describes pagination; `wb_data[1]` contains the records. Different APIs use different envelopes.

Check the reported page count. If the query expands beyond one page, collect the remaining pages before calling the dataset complete.

In [ ]:
metadata, observations = wb_data
if int(metadata["pages"]) != 1:
    raise ValueError("Query spans multiple pages; increase page size or collect remaining pages")
if not isinstance(observations, list):
    raise ValueError("Expected a list of indicator observations")
print("Observations:", len(observations))
if observations:
    print(json.dumps(observations[0], indent=2))

### Step 3 — Create a country–year table
Extract nested country names, the year, and the reported value. A null value is missing, not zero. Keep country codes for joining.

In [ ]:
population_df = pd.DataFrame([{
    "country_code": obs["countryiso3code"], "country": obs["country"]["value"],
    "year": int(obs["date"]), "population": obs["value"]
} for obs in observations])
population_df["population"] = pd.to_numeric(population_df["population"], errors="coerce")
if population_df.duplicated(["country_code", "year"]).any():
    raise ValueError("Duplicate country-year observations")
display(population_df.sort_values(["country_code", "year"]))

In [ ]:
population_wide = population_df.pivot(index="year", columns="country_code", values="population").sort_index()
display(population_wide)
(population_wide / 1_000_000).plot(marker="o")
plt.ylabel("Population (millions)")
plt.xlabel("Year")
plt.title("World Bank total population indicator")
plt.xticks(population_wide.index)
plt.tight_layout()
plt.show()

## Working reliably with APIs
### Common problems
| Problem | What to check |
|---|---|
| `404` | Endpoint path or resource name |
| `400` | Required parameters and accepted values |
| `401` | Missing or invalid credentials |
| `403` | Access restrictions or provider rate limits |
| `429` | Request rate; follow provider retry guidance |
| `500` / `503` | Temporary service problem |
| JSON parsing error | Response may be HTML or empty |
| Missing field | Schema or variable selection |

Check status **before** extracting JSON. An error can itself be valid JSON. A successful status still requires checking fields, units, and completeness.

### Authentication
Some APIs require an API key or bearer token. Read credentials from the environment and send them through the documented mechanism. Never write tokens directly in a shared notebook or print authorization headers.

# See you next lecture!